In [140]:
import re
from collections import Counter
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [141]:
file_path = "rus.txt"
pairs = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            eng = parts[0]
            rus = parts[1]
            pairs.append((eng, rus))


print(pairs[:20])
print(len(pairs))

[('Go.', 'Марш!'), ('Go.', 'Иди.'), ('Go.', 'Идите.'), ('Hi.', 'Здравствуйте.'), ('Hi.', 'Привет!'), ('Hi.', 'Хай.'), ('Hi.', 'Здрасте.'), ('Hi.', 'Здоро́во!'), ('Run!', 'Беги!'), ('Run!', 'Бегите!'), ('Run.', 'Беги!'), ('Run.', 'Бегите!'), ('Who?', 'Кто?'), ('Wow!', 'Вот это да!'), ('Wow!', 'Круто!'), ('Wow!', 'Здорово!'), ('Wow!', 'Ух ты!'), ('Wow!', 'Ого!'), ('Wow!', 'Вах!'), ('Fire!', 'Огонь!')]
363386


In [142]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zа-яё\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_pairs = []

for eng, rus in pairs:
    eng = clean_text(eng)
    rus = clean_text(rus)
    cleaned_pairs.append((eng, rus))

print(cleaned_pairs[:20])

[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здорово'), ('run', 'беги'), ('run', 'бегите'), ('run', 'беги'), ('run', 'бегите'), ('who', 'кто'), ('wow', 'вот это да'), ('wow', 'круто'), ('wow', 'здорово'), ('wow', 'ух ты'), ('wow', 'ого'), ('wow', 'вах'), ('fire', 'огонь')]


In [143]:
processed_pairs = []
for eng, rus in cleaned_pairs:
    rus = "<sos> " + rus + " <eos>"
    processed_pairs.append((eng, rus))

print(processed_pairs[:20])

[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>'), ('hi', '<sos> хай <eos>'), ('hi', '<sos> здрасте <eos>'), ('hi', '<sos> здорово <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('who', '<sos> кто <eos>'), ('wow', '<sos> вот это да <eos>'), ('wow', '<sos> круто <eos>'), ('wow', '<sos> здорово <eos>'), ('wow', '<sos> ух ты <eos>'), ('wow', '<sos> ого <eos>'), ('wow', '<sos> вах <eos>'), ('fire', '<sos> огонь <eos>')]


In [144]:
def build_vocab(sentences):
    counter = Counter()

    for sent in sentences:
        words = sent.split()
        counter.update(words)

    vocab = {"<pad>": 0, "<oov>": 1}

    for word in counter:
        vocab[word] = len(vocab)

    return vocab

In [145]:
eng_sentences = [p[0] for p in processed_pairs]
rus_sentences = [p[1] for p in processed_pairs]

In [146]:
eng_vocab = build_vocab(eng_sentences)
rus_vocab = build_vocab(rus_sentences)

print("English vocabulary size: ", len(eng_vocab))
print("Russian vocabulary size: ", len(rus_vocab))

English vocabulary size:  16319
Russian vocabulary size:  53282


In [147]:
def sentence_to_idx(sentence, vocab):
    if isinstance(sentence, str):
        sentence = sentence.split()

    return [vocab.get(word, vocab["<oov>"]) for word in sentence]

In [148]:
data = []
for eng, rus in processed_pairs:
    eng_idx = sentence_to_idx(eng, eng_vocab)
    rus_idx = sentence_to_idx(rus, rus_vocab)
    data.append((eng_idx, rus_idx))

print(data[:20])

[([2], [2, 3, 4]), ([2], [2, 5, 4]), ([2], [2, 6, 4]), ([3], [2, 7, 4]), ([3], [2, 8, 4]), ([3], [2, 9, 4]), ([3], [2, 10, 4]), ([3], [2, 11, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([5], [2, 14, 4]), ([6], [2, 15, 16, 17, 4]), ([6], [2, 18, 4]), ([6], [2, 11, 4]), ([6], [2, 19, 20, 4]), ([6], [2, 21, 4]), ([6], [2, 22, 4]), ([7], [2, 23, 4])]


In [149]:
train_data,val_data = train_test_split(data,test_size=0.2, random_state=42)

print("Training data size: ", len(train_data))
print("Validation data size: ", len(val_data))

Training data size:  290708
Validation data size:  72678


### Data Loaders Creation

In [150]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [151]:
class TranslationDataset(Dataset):

    def __init__(self, sentence_pairs):
        self.pairs = sentence_pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        return torch.tensor(eng), torch.tensor(rus)

In [152]:
def collate_fn(batch):
    eng_batch, rus_batch = [], []

    for eng, rus in batch:
        eng_batch.append(eng)
        rus_batch.append(rus)

    eng_padded = pad_sequence(eng_batch, padding_value=0)
    rus_padded = pad_sequence(rus_batch, padding_value=0)

    return eng_padded, rus_padded

In [153]:
train_dataset = TranslationDataset(train_data)
val_dataset = TranslationDataset(val_data)

In [154]:
batch_size = 32

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          collate_fn=collate_fn)

val_loader = DataLoader(val_dataset,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=collate_fn)

In [155]:
for eng_batch, rus_batch in train_loader:
    print(eng_batch.shape)
    print(rus_batch.shape)
    break

torch.Size([16, 32])
torch.Size([14, 32])


### Model Architecture

In [156]:
import torch.nn as nn


class Encoder(nn.Module):

    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()

        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell

In [157]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1

        return outputs

In [158]:
input_dim = len(eng_vocab)
output_dim = len(rus_vocab)

emb_dim = 256
hidden_dim = 1024
num_layers = 1

### Training

In [159]:
encoder = Encoder(input_dim, emb_dim, hidden_dim, num_layers)
decoder = Decoder(output_dim, emb_dim, hidden_dim, num_layers)

In [160]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq(encoder, decoder, device).to(device)

In [161]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)
pad_idx = rus_vocab["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

In [162]:
def train(model, loader, optimizer, criterion, clip, device):

    model.train()
    epoch_loss = 0

    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [163]:
def evaluate(model, loader, criterion, device):

    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, teacher_forcing_ratio=0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].reshape(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [164]:
import time

num_epochs = 10
clip = 1

best_val_loss = float('inf')

for epoch in range(num_epochs):
    start = time.time()
    train_loss = train(model, train_loader, optimizer, criterion, clip, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    end = time.time()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'best_model_{epoch}.pt')

    print(f"Epoch {epoch+1}")
    print(f"Train loss: {train_loss}")
    print(f"Val loss: {val_loss}")

Epoch 1
Train loss: 3.502577837811241
Val loss: 2.887019557694734
Epoch 2
Train loss: 2.0173331161285404
Val loss: 2.5914793418536726
Epoch 3
Train loss: 1.546307832071528
Val loss: 2.5460400739291185
Epoch 4
Train loss: 1.2811346857837747
Val loss: 2.5563819178603064
Epoch 5
Train loss: 1.1271598233845972
Val loss: 2.6026149433053716
Epoch 6
Train loss: 1.0297106555450337
Val loss: 2.630908243777886
Epoch 7
Train loss: 0.9596692218712286
Val loss: 2.689044909997725
Epoch 8
Train loss: 0.9027941356698562
Val loss: 2.725216087163754
Epoch 9
Train loss: 0.8624668633996883
Val loss: 2.7473855209077747
Epoch 10
Train loss: 0.8281964540547175
Val loss: 2.80197535591646


In [171]:
import torch
import torch.nn.functional as F

def translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=5, max_len=50):

    model.eval()

    if isinstance(sentence, str):
        tokens = sentence.lower().split()
    else:
        tokens = sentence

    src_indexes = [eng_vocab.get(t, eng_vocab["<oov>"]) for t in tokens]

    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    beams = [([rus_vocab["<sos>"]], 0.0, hidden, cell)]

    for _ in range(max_len):

        new_beams = []

        for seq, score, h, c in beams:

            last_token = seq[-1]

            if last_token == rus_vocab["<eos>"]:
                new_beams.append((seq, score, h, c))
                continue

            input_tensor = torch.LongTensor([last_token]).to(device)

            with torch.no_grad():
                output, h_new, c_new = model.decoder(input_tensor, h, c)

            probs = F.log_softmax(output, dim=1)

            top_probs, top_idx = probs.topk(beam_size)

            for i in range(beam_size):

                token = top_idx[0][i].item()
                token_prob = top_probs[0][i].item()

                new_seq = seq + [token]
                new_score = score + token_prob

                new_beams.append((new_seq, new_score, h_new, c_new))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

    best_seq = beams[0][0]

    inv_vocab = {v:k for k,v in rus_vocab.items()}

    translation = [inv_vocab[i] for i in best_seq]

    return translation[1:]

In [174]:
sentences = [
    "hello how are you",
    "i love programming",
    "where is the cinema",
    "this is a good idea",
    "please show me the path",
    "can you help me"
]

for sentence in sentences:

    translation = translate_beam_search(
        sentence,
        model,
        eng_vocab,
        rus_vocab,
        device,
        beam_size=5
    )
    print("English:", sentence)
    print("Russian:", " ".join(translation))
    print()

English: hello how are you
Russian: как ты <eos>

English: i love programming
Russian: я люблю тома <eos>

English: where is the cinema
Russian: где находится <eos>

English: this is a good idea
Russian: хорошая идея <eos>

English: please show me the path
Russian: пожалуйста покажите мне дорогу <eos>

English: can you help me
Russian: вы можете мне помочь <eos>

